In [2]:
import json

input_path = "datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"
output_path = "datasets/scenes_enriched_error/train_error_scenes_enriched_fixed.jsonl"

written = 0
skipped = 0

with open(input_path, "r", encoding="utf-8") as f_in, \
     open(output_path, "w", encoding="utf-8") as f_out:

    buffer = ""

    for i, raw_line in enumerate(f_in, 1):
        raw_line = raw_line.strip()
        if not raw_line:
            continue

        # Append to buffer
        buffer += raw_line

        # Try to decode all complete JSON objects from the buffer
        while buffer:
            try:
                obj, end_idx = json.JSONDecoder().raw_decode(buffer)
                f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
                written += 1
                buffer = buffer[end_idx:].strip()
            except json.JSONDecodeError:
                break  # incomplete object, wait for more lines

    # Anything left in buffer is unrecoverable
    if buffer.strip():
        print(f"[SKIP] Unrecoverable leftover: {repr(buffer[:200])}")
        skipped += 1

print(f"\nDone. Written: {written}, Skipped: {skipped}")
print(f"Expected ~10811, Got: {written}")


Done. Written: 5799, Skipped: 0
Expected ~10811, Got: 5799


In [3]:
import json

input_path = "datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"

# Just inspect the first 5 lines raw
with open(input_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(f"\n=== Line {i} ===")
        print(f"Length: {len(line)}")
        print(f"Content: {repr(line[:100])}")
        print(f"Ends with: {repr(line[-20:])}")
        
        # Count how many JSON objects are on this line
        decoder = json.JSONDecoder()
        pos = 0
        count = 0
        line = line.strip()
        while pos < len(line):
            try:
                obj, end_idx = decoder.raw_decode(line, pos)
                count += 1
                pos = end_idx
                # skip whitespace
                while pos < len(line) and line[pos] in ' \t\r\n':
                    pos += 1
            except json.JSONDecodeError:
                print(f"  Parse stopped at pos {pos}: {repr(line[pos:pos+20])}")
                break
        print(f"Objects found on this line: {count}")
        
        if i >= 5:
            break


=== Line 1 ===
Length: 1270
Content: '{"script_id": "synthetic_script_0010_error", "scene_id": "synthetic_script_0010_error_scene_01", "sc'
Ends with: '_previous": false}}\n'
Objects found on this line: 1

=== Line 2 ===
Length: 1174
Content: '{"script_id": "synthetic_script_0010_error", "scene_id": "synthetic_script_0010_error_scene_02", "sc'
Ends with: '_previous": false}}\n'
Objects found on this line: 1

=== Line 3 ===
Length: 1194
Content: '{"script_id": "synthetic_script_0010_error", "scene_id": "synthetic_script_0010_error_scene_03", "sc'
Ends with: '_previous": false}}\n'
Objects found on this line: 1

=== Line 4 ===
Length: 1157
Content: '{"script_id": "synthetic_script_0010_error", "scene_id": "synthetic_script_0010_error_scene_04", "sc'
Ends with: '_previous": false}}\n'
Objects found on this line: 1

=== Line 5 ===
Length: 877
Content: '{"script_id": "synthetic_script_0010_error", "scene_id": "synthetic_script_0010_error_scene_05", "sc'
Ends with: '_previous": false}}\n'
O

In [4]:
import json

input_path = "datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl"

total_lines = 0
empty_lines = 0
parse_ok = 0
parse_fail = 0

with open(input_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        total_lines += 1
        stripped = line.strip()
        
        if not stripped:
            empty_lines += 1
            continue
        
        try:
            json.loads(stripped)
            parse_ok += 1
        except json.JSONDecodeError as e:
            parse_fail += 1
            if parse_fail <= 10:  # show first 10 failures
                print(f"Line {i}: col {e.colno} | {repr(stripped[max(0,e.colno-10):e.colno+10])}")

print(f"\nTotal lines  : {total_lines}")
print(f"Empty lines  : {empty_lines}")
print(f"Parse OK     : {parse_ok}")
print(f"Parse FAIL   : {parse_fail}")
print(f"OK + FAIL    : {parse_ok + parse_fail}")


Total lines  : 5799
Empty lines  : 0
Parse OK     : 5799
Parse FAIL   : 0
OK + FAIL    : 5799


In [6]:
# Count actual JSON objects (not raw lines)
#python3 -c "
import json
count = 0
errors = 0
with open('datasets/scenes_enriched_error/train_error_scenes_enriched.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            json.loads(line)
            count += 1
        except:
            errors += 1
print(f'Valid JSON objects: {count}')
print(f'Parse errors: {errors}')
#"

Valid JSON objects: 5799
Parse errors: 0
